# 提交说明（Chapter 4）

本 notebook 用于展示治理后数据在预测任务中的衔接。

- 运行前请先执行参数区。
- 提交前请清理输出与敏感路径。

In [ ]:
# 参数区（提交版本）
import os
from pathlib import Path

DATA_ROOT = Path(os.environ.get("CH5_DATA_ROOT", "./data_sample"))
OUTPUT_ROOT = Path(os.environ.get("CH5_OUTPUT_ROOT", "./outputs"))
MODEL_ROOT = Path(os.environ.get("CH5_MODEL_ROOT", "./models"))

for p in [DATA_ROOT, OUTPUT_ROOT, MODEL_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("MODEL_ROOT:", MODEL_ROOT)

In [ ]:
### Market Forecast
### Eric Wu, GM Office, BCH China
### Jul. 2024

In [ ]:
import pandas as pd
from prophet import Prophet

from prophet.diagnostics import cross_validation, performance_metrics
import itertools
import numpy as np

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
path = r'//bcnshgs0295/02_FlatDataSource/01_UploadFile/82_GM_Office/02_Data_Platform/09_Reports/Forecast/Raw_Data/'

In [ ]:
channel = 'Online' # Online
df_Category = pd.read_excel(path + 'Category_Template.xlsx',sheet_name = channel)
#df_Brand = pd.read_csv(path + 'Brand.csv')

In [ ]:
events = pd.DataFrame({
    'holiday': 'covid',
    'ds': pd.to_datetime(['2022-11-01', '2022-12-01']),
    'lower_window': 0,
    'upper_window': 1,
})

def prepare_data(df, value_col):
    df_temp = df[['Date', 'Channel', 'Global_Category', value_col]].copy()
    df_temp = df_temp.rename(columns={'Date': 'ds', value_col: 'y'})
    df_temp['ds'] = pd.to_datetime(df_temp['ds'])
    return df_temp[['ds', 'y']]

param_grid = {
    'changepoint_prior_scale': [0.01,0.1],
    'seasonality_prior_scale': [0.01,0.1],
}


all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]

In [ ]:
from tqdm import tqdm

def optimize_and_forecast(df, periods= 5 + 12*5, freq='MS', start_date='2025-08-01'): # Update the periods and start_date
    predictions = []
    
    total_iterations = len(df['Channel'].unique()) * len(df['Global_Category'].unique()) * len(all_params)
    
    with tqdm(total=total_iterations, desc="Optimizing Parameters") as pbar:
        for channel in df['Channel'].unique():
            for category in df['Global_Category'].unique():
                df_filtered = df[(df['Channel'] == channel) & (df['Global_Category'] == category)]
                if df_filtered.empty:
                    pbar.update(len(all_params))
                    continue
                df_prepared = prepare_data(df_filtered, 'Values')
                
                best_params = None
                best_rmse = float('inf')
                
                for params in all_params:
                    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                                seasonality_mode='multiplicative',
                                changepoint_prior_scale=params['changepoint_prior_scale'],
                                seasonality_prior_scale=params['seasonality_prior_scale'],
                                holidays=events)
                    
                    m.fit(df_prepared)
                    
                    try:
                        cv_results = cross_validation(m, initial='365 days', 
                                                      period='90 days', 
                                                      horizon='180 days',
                                                      parallel='processes')
                        df_p = performance_metrics(cv_results)
                        rmse = df_p['rmse'].mean()
                        
                        if rmse < best_rmse:
                            best_rmse = rmse
                            best_params = params
                    except Exception as e:
                        print(f"Error during cross-validation for {channel} - {category}: {e}")
            
                    pbar.update(1)
                
                if best_params:
                    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                                seasonality_mode='additive',#additive,multiplicative
                                changepoint_prior_scale=best_params['changepoint_prior_scale'],
                                seasonality_prior_scale=best_params['seasonality_prior_scale'],
                                holidays=events)
                    
                    m.fit(df_prepared)
                    future = m.make_future_dataframe(periods=periods, freq=freq)
                    forecast = m.predict(future)
                    forecast = forecast[forecast['ds'] >= start_date]
                    forecast['Channel'] = channel
                    forecast['Global_Category'] = category
                    forecast.rename(columns={'ds': 'Date', 'yhat': 'Values'}, inplace=True)
                    predictions.append(forecast[['Date', 'Channel', 'Global_Category', 'Values']])
    
    return pd.concat(predictions)

### Category

In [ ]:
df_Category['Date'] = pd.to_datetime(df_Category['Date'], format='%Y%m')
df_Category


In [ ]:
#df_Category_Process = df_Category.groupby(['Date', 'Global_Category'])['Values'].sum().reset_index()
#df_Category_Process['Channel'] = 'TTL'
df_Category_Process = df_Category.copy()

In [ ]:
df_Predicted_Category = optimize_and_forecast(df_Category_Process)
df_Predicted_Category_Results = pd.concat([df_Category_Process, df_Predicted_Category]).reset_index(drop=True)

print(df_Predicted_Category_Results)

In [ ]:
df_Predicted_Category_Results['Date'] = pd.to_datetime(df_Predicted_Category_Results['Date'])
df_Predicted_Category_Results.to_excel(path + 'Forecast_Category.xlsx'.format(channel),index=False)

In [ ]:
### END

### Brand

In [ ]:
df_Brand_Process = df_Brand.groupby(['Date', 'Global_Category'])['Values'].sum().reset_index()
df_Brand_Process['Channel'] = 'TTL'

In [ ]:
df_predicted_Brand = optimize_and_forecast(df_Brand_Process)
df_predicted_Brand_Results = pd.concat([df_Brand_Process, df_predicted_Brand]).reset_index(drop=True)

In [ ]:
df_predicted_Brand_Results['Date'] = pd.to_datetime(df_predicted_Brand_Results['Date'])
df_predicted_Brand_Results.to_excel(path + 'Forecast_Brand.xlsx',index=False)

### Integration

In [ ]:
df_Predicted_Category_Results = df_Predicted_Category_Results.rename(columns={'Values': 'Category_Size'})
df_predicted_Brand_Results = df_predicted_Brand_Results.rename(columns={'Values': 'Offtake'})

In [ ]:
merged_df = pd.merge(df_Predicted_Category_Results, df_predicted_Brand_Results, on=['Date', 'Global_Category'])

In [ ]:
merged_df.to_excel(path + 'Forecast_Category_Brand.xlsx',index=False)